## Retired setup diagnostic - engineering fixture only
**Metadata-only diagnostics. Never enter an API key in this notebook.** Select the **Python** runtime, not PySpark. Keep the existing workspace and approved Key Vault managed private endpoint. No Lakehouse or Spark Environment is required.

Change only `VAULT_URL` below to the deployed Key Vault HTTPS URL (not a private IP). The single code cell lists secret metadata without reading any value and rejects an existing disabled, expired, or not-yet-valid secret.

There are no input controls, callbacks, secret writes, or secret-value reads. No roles, networking, workload assets or parameters are changed. This historical fixture is not deployed by Fast Pass. The supported credential step uses Azure Custom deployment with a secure ARM parameter.

The hidden-input experiment did not render an input box, and the Password-widget experiment persisted its nonsecret input. Both experiments are retired. Do not add notebook input controls, literals, parameters, or a secret writer.


In [ ]:
import http.client
import json
import re
import time
from urllib.parse import urlsplit
import notebookutils

VAULT_URL = 'https://<vault-name>.vault.azure.net/'
SECRET_NAME = 'geoanalytics-api-key'
if not re.fullmatch(r'https://[A-Za-z][A-Za-z0-9-]{1,22}[A-Za-z0-9]\.vault\.azure\.net/', VAULT_URL):
    raise ValueError('Set VAULT_URL to the deployed https://<vault-name>.vault.azure.net/ URL. No key belongs in this cell.')

def secret_metadata_present() -> bool:
    url = VAULT_URL + 'secrets?api-version=7.4'
    seen = set()
    token = None
    try:
        token = notebookutils.credentials.getToken('keyvault')
        while url:
            parsed = urlsplit(url)
            if (parsed.scheme != 'https' or parsed.netloc != urlsplit(VAULT_URL).netloc
                    or parsed.path != '/secrets' or parsed.fragment or url in seen):
                raise RuntimeError('Unexpected Key Vault metadata continuation URL. Stopped without following it.')
            seen.add(url)
            connection = http.client.HTTPSConnection(parsed.netloc, timeout=30)
            try:
                connection.request('GET', parsed.path + '?' + parsed.query, headers={'Authorization': 'Bearer ' + token})
                response = connection.getresponse()
                if response.status != 200:
                    raise RuntimeError(f'Key Vault metadata check HTTP {response.status}. Verify private connectivity and secret list permission; no key was requested by this check.')
                page = json.loads(response.read())
            finally:
                connection.close()
            if not isinstance(page, dict) or not isinstance(page.get('value'), list):
                raise RuntimeError('Invalid Key Vault metadata response; setup stopped.')
            for item in page['value']:
                if item.get('id', '').rstrip('/') == VAULT_URL + 'secrets/' + SECRET_NAME:
                    attributes = item.get('attributes', {})
                    now = time.time()
                    if (attributes.get('enabled') is not True
                            or (attributes.get('exp') is not None and attributes['exp'] <= now)
                            or (attributes.get('nbf') is not None and attributes['nbf'] > now)):
                        raise RuntimeError('Existing secret is disabled, expired, or not yet valid. No replacement was written; contact the Key Vault owner.')
                    return True
            url = page.get('nextLink')
        return False
    except (OSError, http.client.HTTPException, ValueError):
        raise RuntimeError('Key Vault metadata check failed. Verify private DNS/connectivity and the Python runtime; do not enter a key or expose diagnostics containing credentials.') from None
    finally:
        token = None

_secret_present = secret_metadata_present()
print('Enabled geoanalytics-api-key metadata verified. No value retrieved; runtime access is not yet validated.' if _secret_present else 'Metadata check passed; secret is absent. Secret creation requires an approved private management surface outside this notebook.')
